In [ ]:
import gc
import importlib
import importlib.util
import inspect
import logging
import sys
import time
import traceback
import zipfile
from pathlib import Path

import numpy as np
import torch


# CHECK T4 GPU, INSTALL LIBRARIES & CLEAN EXTRACT
if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab is running on CPU! Please go to Runtime -> Change runtime type -> select T4 GPU and run again."
    )

zip_path = Path("eval_for_colab.zip")
if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
    raise RuntimeError("Valid 'eval_for_colab.zip' file not found. Please upload and wait for 100% completion!")

print(f">>> GPU detected: {torch.cuda.get_device_name(0)}. Installing libraries...")
!pip install -q chromadb rank-bm25 pyyaml langchain-community > /dev/null 2>&1

!rm -rf "Evaluation" "Retrieval" "Query Understanding and Routing" "Document Ingestion and Parsing" eval_results.zip
!unzip -q -o eval_for_colab.zip
!if [ -f output_ready.zip ]; then unzip -q -o output_ready.zip; fi

for req_dir in (
    "Document Ingestion and Parsing/output/Ingestion",
    "Document Ingestion and Parsing/output/Chunking",
    "Document Ingestion and Parsing/output/Provenance",
    "Document Ingestion and Parsing/output/Vector DB",
):
    if not Path(req_dir).is_dir():
        raise FileNotFoundError(
            f"Missing directory '{req_dir}'. Please re-run 'pack_eval.py --full' on your computer!"
        )

ROOT_DIR = Path.cwd()
RETRIEVAL_DIR = ROOT_DIR / "Retrieval"
CHUNKING_DIR = ROOT_DIR / "Document Ingestion and Parsing" / "Chunking"
for p in (ROOT_DIR, RETRIEVAL_DIR, CHUNKING_DIR):
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

EVAL_DIR = Path("Evaluation/Chunking Evals")


# LAYER 1 · INTRINSIC EVALUATION 
l1_file = EVAL_DIR / "evals_layer1.py"
if not l1_file.is_file():
    raise FileNotFoundError(f"File '{l1_file}' not found. Please check the eval_for_colab.zip archive!")

print("\n" + "=" * 70)
print(f">>> [LAYER 1] Running: {l1_file.name} (BI_atomic, ICC, DCC)")
print("=" * 70)

spec_l1 = importlib.util.spec_from_file_location("evals_layer1_mod", str(l1_file.resolve()))
evals_layer1_mod = importlib.util.module_from_spec(spec_l1)
sys.modules["evals_layer1_mod"] = evals_layer1_mod
spec_l1.loader.exec_module(evals_layer1_mod)

# Dynamic batch splitting (step=8 for long DCC windows > 2000 chars) to avoid overflowing 15GB VRAM on T4
_l1_logger = logging.getLogger("eval_layer1")

def _safe_embed_texts(self, texts, batch_size: int = 32):
    missing = list(dict.fromkeys(t for t in texts if t not in self._memo))
    if missing:
        if self._tok and self._max_len:
            for t in missing:
                if len(self._tok(t, add_special_tokens=True, truncation=False)["input_ids"]) > self._max_len:
                    _l1_logger.warning(
                        "Input exceeds '%s' max_seq_length (%d) and will be truncated.",
                        self.model_name,
                        self._max_len,
                    )
                    break
        idx = 0
        while idx < len(missing):
            step = 8 if any(len(t) > 2000 for t in missing[idx : idx + batch_size]) else batch_size
            batch = missing[idx : idx + step]
            idx += step
            if self._encode_fn is not None:
                arr = np.asarray(self._encode_fn(batch), dtype=np.float32)
            elif self._qv is not None:
                with torch.inference_mode():
                    arr = np.asarray(self._qv.vectorize_passages(batch), dtype=np.float32)
            else:
                with torch.inference_mode():
                    arr = np.asarray(
                        self._st.encode(
                            batch, batch_size=len(batch), normalize_embeddings=True, show_progress_bar=False
                        ),
                        dtype=np.float32,
                    )
            norms = np.linalg.norm(arr, axis=1, keepdims=True)
            arr = arr / np.where(norms > 0, norms, 1.0)
            for txt, vec in zip(batch, arr):
                self._memo[txt] = vec
    return {t: self._memo[t] for t in texts}

if hasattr(evals_layer1_mod, "Embedder"):
    evals_layer1_mod.Embedder.embed_texts = _safe_embed_texts

t_l1 = time.perf_counter()
l1_report_path = EVAL_DIR / "eval_layer1_report.json"
evals_layer1_mod.run_evaluation(
    out_path=l1_report_path,
    strategies=["fixed_size", "structure_aware", "semantic"],
    embed_model="BAAI/bge-m3",
    embed_text="source",
    dcc_window_tokens=1500,
    dcc_window_step=1,
    icc_min_unit_tokens=0,
)
print(f">>> [LAYER 1] Completed in {time.perf_counter() - t_l1:.1f}s -> {l1_report_path}")

# Completely release the Layer 1 model before loading the Layer 2 model
del evals_layer1_mod, sys.modules["evals_layer1_mod"]
gc.collect()
torch.cuda.empty_cache()


# LAYER 2 · EXTRINSIC EVALUATION 
print("\n" + "=" * 70)
print(">>> [LAYER 2] Running: evals_layer2.py (Fixed Retriever @ Top-5)")
print("=" * 70)

from sentence_transformers import CrossEncoder

# Patch CrossEncoder.predict bug on sentence-transformers v5 + transformers v5 (KeyError: 'ne')
def _safe_ce_predict(self, sentences, batch_size: int = 32, show_progress_bar: bool = False, **kwargs):
    if not sentences:
        return []
    dev = next(self.model.parameters()).device
    max_len = getattr(self, "max_length", None) or 512
    all_scores: list[float] = []
    self.model.eval()
    with torch.inference_mode():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i : i + batch_size]
            enc = self.tokenizer(
                [str(p[0]) for p in batch],
                [str(p[1]) for p in batch],
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            )
            enc = {k: v.to(dev) for k, v in enc.items()}
            logits = self.model(**enc).logits.view(-1)
            all_scores.extend(torch.sigmoid(logits).float().cpu().tolist())
    return all_scores

CrossEncoder.predict = _safe_ce_predict

# Reload original Retrieval modules to prevent RecursionError when re-running the cell
import query_vectorizer
import reranker
import retrieval_system
importlib.reload(query_vectorizer)
importlib.reload(reranker)
importlib.reload(retrieval_system)

from query_vectorizer import QueryVectorizer
from reranker import CrossEncoderReranker

# Force QueryVectorizer to run on CUDA FP16 + vector cache for MMR
_orig_qv_get = QueryVectorizer._get_model
def _cuda_qv_get(self):
    self.device = "cuda"
    m = _orig_qv_get(self).to("cuda")
    if not getattr(m, "_is_fp16", False):
        m.half()
        m._is_fp16 = True
    return m

_passage_cache: dict[str, list[float]] = {}
_orig_vec_passages = QueryVectorizer.vectorize_passages
def _cached_vec_passages(self, texts: list[str]) -> list[list[float]]:
    clean = [(t or "").strip() for t in texts]
    missing = [t for t in dict.fromkeys(clean) if t and t not in _passage_cache]
    if missing:
        with torch.inference_mode():
            encoded = _orig_vec_passages(self, missing)
        for txt, vec in zip(missing, encoded):
            _passage_cache[txt] = vec
    return [_passage_cache.get(t, []) if t else [] for t in clean]

QueryVectorizer._get_model = _cuda_qv_get
QueryVectorizer.vectorize_passages = _cached_vec_passages

# Force CrossEncoderReranker to run on CUDA FP16 and share 1 instance (threshold=0.0)
_orig_ce_get = CrossEncoderReranker._get_model
def _cuda_ce_get(self):
    self.device = "cuda"
    ce = _orig_ce_get(self)
    ce.to("cuda")
    if hasattr(ce, "model") and not getattr(ce.model, "_is_fp16", False):
        ce.model.half()
        ce.model._is_fp16 = True
    return ce

CrossEncoderReranker._get_model = _cuda_ce_get

rerank_kwargs = {"threshold": 0.0, "top_k": None}
if "device" in inspect.signature(CrossEncoderReranker.__init__).parameters:
    rerank_kwargs["device"] = "cuda"
_shared_eval_reranker = CrossEncoderReranker(**rerank_kwargs)

_orig_rs_init = retrieval_system.RetrievalSystem.__init__
def _patched_rs_init(self, *args, **kwargs):
    kwargs["reranker"] = _shared_eval_reranker
    _orig_rs_init(self, *args, **kwargs)

retrieval_system.RetrievalSystem.__init__ = _patched_rs_init

# Run evals_layer2.py with ground_truth_chunking.json -> eval_layer2_report.json
gt_file = EVAL_DIR / "ground_truth_chunking.json"
if not gt_file.is_file():
    raise FileNotFoundError(f"File '{gt_file}' not found")

l2_file = EVAL_DIR / "evals_layer2.py"
if not l2_file.is_file():
    raise FileNotFoundError(f"File '{l2_file}' not found")

l2_code = l2_file.read_text(encoding="utf-8")
l2_code = l2_code.replace("CURRENT_DIR.parents[2]", "CURRENT_DIR.parents[1]")
l2_file.write_text(l2_code, encoding="utf-8")

t_l2 = time.perf_counter()
try:
    sys.argv = [str(l2_file)]
    exec(compile(l2_code, str(l2_file), "exec"), {"__name__": "__main__", "__file__": str(l2_file.resolve())})
except Exception:
    traceback.print_exc()
    raise
print(f">>> [LAYER 2] Completed in {time.perf_counter() - t_l2:.1f}s -> {EVAL_DIR / 'eval_layer2_report.json'}")

# COMPRESS ALL JSON RESULTS AND DOWNLOAD TO LAPTOP
!zip -q -r eval_results.zip "Evaluation/Chunking Evals" -x "*__pycache__*" "*.cache_layer1*"
from google.colab import files
files.download("eval_results.zip")